# Varying n

In [ ]:
import numpy as np
from MyData import load_xg_dataset
from MeasureBias import equibucket_bias
from UnbiasedBinning import unbiased_binning
from EbiasedBinning import ebias_binning
from EbiasedDnC import ebias_dnc
from time import time
import tracemalloc


D = load_xg_dataset('datasets/german_credit_data.csv', x_idx=7, g_idx=2)
n = D.shape[0]; k=3
NoCases = 5
exec_space_u = np.zeros(NoCases,dtype=float)
exec_space_e = np.zeros(NoCases,dtype=float)
exec_space_dnc = np.zeros(NoCases,dtype=float)
exec_space_ls = np.zeros(NoCases,dtype=float)

rep=1
for i in range(NoCases):
    nprime = int((n/NoCases)*(i+1))
    equisize = int(nprime/k)
    np.random.seed(rep)
    idx = np.random.choice(D.shape[0], nprime, replace=True)
    D_sample = D[idx]

    tracemalloc.start()
    result = unbiased_binning(D_sample,k=k,x=0,groups=1)
    _, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    exec_space_u[i] = peak_bytes / (1024 ** 2)

    tracemalloc.start()
    result = ebias_binning(D_sample,k=k,eps=0.03)
    _, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    exec_space_e[i] = peak_bytes / (1024 ** 2)
    
    tracemalloc.start()
    result = ebias_dnc(D_sample,k=k,eps=0.03,fast=True)
    _, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    exec_space_dnc[i] = peak_bytes / (1024 ** 2)

    tracemalloc.start()
    result = ebias_dnc(D_sample,k=k,eps=0.03,fast=False)
    _, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    exec_space_ls[i] = peak_bytes / (1024 ** 2)



/Users/adam/Git/UnbiasedBinningPrivate/Python/MyData.py:75: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  g = g_raw.replace(g_mapping).values


0.010815620422363281
0.33602142333984375
0.009387969970703125
0.00836944580078125
0.012763023376464844
1.2651071548461914
0.014418601989746094
0.013263702392578125
0.0167999267578125
2.8123178482055664
0.019576072692871094
0.019252777099609375
0.0213775634765625
4.971695899963379
0.026686668395996094
0.025524139404296875
0.0259552001953125
7.737858772277832
0.031360626220703125
0.03165435791015625


In [ ]:
n = 1000 #D.shape[0]; k=3
print(n)
# print(exec_space_u)
# print(exec_space_e)
# print(exec_space_dnc)
# print(exec_space_ls)
'''
exec_space_u: [0.01081562, 0.01276302, 0.01679993, 0.02137756, 0.0259552 ]
exec_space_e:[0.33602142, 1.26510715, 2.81231785, 4.9716959, 7.73785877]
exec_space_dnc: [0.00938797, 0.0144186, 0.01957607, 0.02668667, 0.03136063]
exec_space_ls:[0.00836945, 0.0132637, 0.01925278, 0.02552414, 0.03165436]
'''

1000


In [5]:
# Varying n
import matplotlib.pyplot as plt
import numpy as np

NoCases = 5; n=1000
xaxis = [int((n/NoCases)*(i+1)) for i in range(NoCases)]


# Plot settings
plt.figure(figsize=(5, 4))  # Small for multi-column layout
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.plot(xaxis, exec_space_e, marker='s', color='#ff7f0e', linewidth=2, markersize=6, label='DP: $\epsilon$-biased ($\epsilon$=0.03)')
plt.plot(xaxis, exec_space_ls, marker='^', color='#2ca02c', linewidth=2, markersize=6, label='LS: $\epsilon$-biased ($\epsilon$=0.03)')
plt.plot(xaxis, exec_space_dnc, marker='x', color='#9467bd', linewidth=2, markersize=6, label='D&C: $\epsilon$-biased ($\epsilon$=0.03)')
plt.plot(xaxis, exec_space_u, marker='o', color='#1f77b4', linewidth=2, markersize=6, label='Unbiased Binning')
plt.legend(
    fontsize=12,
    loc='right',  # or 'center right'
    bbox_to_anchor=(1.025, 0.62),  # x stays at the right edge, y moves up
    bbox_transform=plt.gca().transAxes,
)


plt.xlabel('Input Size (n)', fontsize=14)
plt.ylabel('Memory Peak (MB)', fontsize=14)
# plt.ylim(0, max(exec_time_b) * 1.05)
plt.yscale('log')
plt.savefig('exp/combined/germancredit/space_vs_n.pdf', format='pdf', bbox_inches='tight')
plt.clf()

<Figure size 500x400 with 0 Axes>